# Ders 02 - Microsoft Agent Framework'ü Keşfetmek

**Microsoft Agent Framework (MAF)**, AI ajanları oluşturmak için birleşik bir çerçevedir. Dört temel yapı taşı ile temiz ve bileşik bir mimari sunar:

- **Client** – bir AI model uç noktasıyla bağlanır ve iletişimi yönetir
- **Agent** – bir istemciyi talimatlar ve araç tanımları ile sarar
- **Tools** – modelin çağırabileceği özel fonksiyonlarla ajan yeteneklerini genişletir
- **Session** – çoklu dönüşlü etkileşimler için konuşma geçmişini korur

Bu derste, bu kavramları kullanarak hedef erişilebilirliğini kontrol eden bir **seyahat rezervasyon ajanı** oluşturacağız.


## Kurulum


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Agent Framework Mimarisi Anlama

Microsoft Agent Framework katmanlı bir mimariyi takip eder:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **İstemci** – Bir `AzureAIProjectAgentProvider`, bir Azure OpenAI dağıtımına bağlanır. Kimlik doğrulama, istek biçimlendirme ve yanıt ayrıştırma işlemlerini yönetir.
2. **Agent** – İstemciden `provider.create_agent()` ile oluşturulur, agent model erişimini talimatlar (sistem istemi) ve araçlarla birleştirir.
3. **Araçlar** – Agent’in eylem gerçekleştirmek veya veri almak için çağırabileceği `@tool` ile süslenmiş Python fonksiyonları.
4. **Oturum** – Konuşma geçmişini depolayan (çok turlu diyalogu mümkün kılan) ve agent’ın önceki bağlamı hatırlamasını sağlayan bir `AgentSession` nesnesi (`agent.create_session()` ile oluşturulur).

Her katmanı adım adım inşa edelim.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool Dekoratörü ile Araç Ekleme

Araçlar, ajanların metin üretmenin ötesinde eylemler yapmasını sağlar. `@tool` dekoratörü, normal bir Python fonksiyonunu ajanın çağırabileceği bir şeye dönüştürür.

Temel noktalar:
- Modelin her parametreyi anlaması için `Annotated[type, "description"]` kullanın.
- Dokümantasyon dizisi, modelin gördüğü araç açıklaması olur.
- `approval_mode="never_require"` aracı, kullanıcı onayı olmadan otomatik olarak çalıştırır.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Araçlarla Bir Ajan Oluşturma

Şimdi istemciyi, talimatları ve araçları bir ajanda birleştiriyoruz. `instructions` sistem istemi olarak görev yapar — ajanının kişiliğini ve davranışını tanımlar.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Oturumlarla Çok Turlu Konuşmalar

Bir `AgentSession` ( `agent.create_session()` ile oluşturulan ), bir konuşmadaki tüm mesajları takip eder. Aynı oturumu her `agent.run()` çağrısına geçirerek, ajan tüm konuşma geçmişine erişebilir ve önceki mesajlara atıfta bulunabilir.

Her turda ajanın kullanılabilirlik denetleyicimizi arayabilmesi için `tools=[check_destination_availability]` parametresini geçiriyoruz.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Özet

Bu derste Microsoft Agent Framework'ün dört temel direğini incelediniz:

| Kavram | Öğrendikleriniz |
|---------|------------------|
| **İstemci** | `AzureAIProjectAgentProvider` kimlik bilgilerine dayalı kimlik doğrulamasıyla Azure OpenAI'ye bağlanır |
| **Ajan** | `provider.create_agent()` model bağlantısını talimatlar ve bir isimle paketler |
| **Araçlar** | `@tool` dekoratörü, ajanın çağırması için Python fonksiyonlarını açığa çıkarır |
| **Oturum** | `agent.create_session()` konuşma geçmişini birden çok tur boyunca korur |

Bu yapı taşları, doğal konuşmalar yapabilen, dış fonksiyonları çağırabilen ve bağlamı sürdürebilen ajanlar oluşturmak için bir araya gelir — daha ileri ajan kalıplarının temelini sonraki derslerde oluşturur.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:  
Bu belge, AI çeviri servisi [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba sarf etsek de, otomatik çevirilerin hatalar veya yanlışlıklar içerebileceğini lütfen unutmayın. Orijinal belge, kendi ana dilindeki haliyle yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi önerilir. Bu çevirinin kullanımıyla ortaya çıkabilecek herhangi bir yanlış anlama veya yanlış yorumdan sorumlu değiliz.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
